# OCR Module - Text Extraction


Bu notebook, görsel dokümanlardan metin çıkarımı (OCR) için yapılan deneyleri içermektedir. Farklı OCR modelleri değerlendirilmiş ve sonuçta **Chandra OCR** modeli seçilmiştir.


# OCR Module - Text Extraction
## CSE 655 - Deep Learning Final Project


Bu notebook, görsel dokümanlardan metin çıkarımı (OCR) için yapılan deneyleri içermektedir. Farklı OCR modelleri değerlendirilmiş ve sonuçta **Chandra OCR** modeli seçilmiştir.

---

### 🔍 Değerlendirilen Modeller

| Model | Avantaj | Dezavantaj | Sonuç |
|-------|---------|------------|-------|
| DeepSeek OCR | Hızlı | Tekrarlı kelimeler, tutarsız çıktı | ❌ |
| HunyuanOCR | Stabil çıktı | Yüksek işlem süresi, kısmen tutarsız çıktı | ❌ |
| **Chandra OCR** | Kalite-hız dengesi, tutarlı çıktı | - | ✅ Seçildi |


> ⚠️ **Not:** Bu notebook yalnızca OCR model seçimi için yapılan deneyleri içermektedir. Tam pipeline için `pipeline.ipynb` dosyasına bakınız.

---

## 1. Chandra OCR Model

In [ ]:
!pip install chandra-ocr --quiet
print("✅ Chandra OCR kuruldu!")


In [ ]:
import torch
from transformers import AutoModelForVision2Seq, AutoProcessor
from PIL import Image
import os

print("🚀 Chandra modeli yükleniyor...")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"📍 Device: {device}")

model = AutoModelForVision2Seq.from_pretrained(
    "datalab-to/chandra",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    trust_remote_code=True,
    device_map="auto"
).eval()

processor = AutoProcessor.from_pretrained(
    "datalab-to/chandra",
    trust_remote_code=True
)

print("✅ Model yüklendi!")


In [ ]:
def chandra_ocr(image_path):
    """Chandra OCR - çalışan versiyon"""

    # Görsel yükle
    image = Image.open(image_path).convert("RGB")

    # Prompt
    prompt = "Extract all text from this image."

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=[text],
        images=[image],
        padding=True,
        return_tensors="pt",
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=4096,
            temperature=0.0,
            do_sample=False,
            use_cache=True,
        )

    generated_ids_trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs["input_ids"], generated_ids)
    ]

    result = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0]

    return result

Test Example

In [ ]:
import os
test_image = "/content/drive/MyDrive/deep-learning/rvlcdip_split_v3/test/form/0001215128.tif"

print(f"📄 Dosya: {os.path.basename(test_image)}")
print("⏳ OCR işleniyor...")

result = chandra_ocr(test_image)

print("\n" + "="*60)
print("📄 SONUÇ:")
print("="*60)
print(result)
print("="*60)

##Hunyuan OCR

In [ ]:
!pip install git+https://github.com/huggingface/transformers@82a06db03535c49aa987719ed0746a76093b1ec4

In [ ]:
import transformers
print(f"Transformers version: {transformers.__version__}")

# Import test
from transformers import HunYuanVLForConditionalGeneration
print("✅ HunYuanVLForConditionalGeneration import başarılı!")

In [ ]:
!pip -q install accelerate safetensors pillow


Test Example - Tutarsız Çıktı

In [ ]:
from transformers import AutoProcessor
from transformers import HunYuanVLForConditionalGeneration
from PIL import Image
import torch

model_name_or_path = "tencent/HunyuanOCR"
processor = AutoProcessor.from_pretrained(model_name_or_path, use_fast=False)
img_path = "/content/drive/MyDrive/deep-learning/rvlcdip_split_v3/test/form/0001215128.tif"
image_inputs = Image.open(img_path)
messages1 = [
    {"role": "system", "content": ""},
    {
        "role": "user",
        "content": [
            {"type": "image", "image": img_path},
            {"type": "text", "text": (
                """Extract all information include blue stamps from the main body of the document image and represent it in markdown format, ignoring headers and footers. Tables should be expressed in HTML format, formulas in the document should be represented using LATEXformat, and the parsing should be organized according to the reading order."""
            )},
        ],
    }
]
messages = [messages1]
texts = [
    processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
    for msg in messages
]
inputs = processor(
    text=texts,
    images=image_inputs,
    padding=True,
    return_tensors="pt",
)
model = HunYuanVLForConditionalGeneration.from_pretrained(
    model_name_or_path,
    attn_implementation="eager",
    dtype=torch.bfloat16,
    device_map="auto"
)
with torch.no_grad():
    device = next(model.parameters()).device
    inputs = inputs.to(device)
    generated_ids = model.generate(**inputs, max_new_tokens=16384, do_sample=False)
if "input_ids" in inputs:
    input_ids = inputs.input_ids
else:
    print("inputs: # fallback", inputs)
    input_ids = inputs.inputs
generated_ids_trimmed = [
    out_ids[len(in_ids):] for in_ids, out_ids in zip(input_ids, generated_ids)
]
output_texts = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(output_texts)

HunyuanOCR-yukarıdaki örnekte olduğu gibi- bazı dokümanlarda metni çıkarıyor ancak tablo yapısını koruyamadığı için çıktıda aşırı tekrar ve boş satır oluşuyor; bu da okunabilirliği ve sonraki analiz kalitesini düşürüyor.

##Deepseek OCR

In [ ]:
# DeepSeek OCR kurulumu
!pip install transformers==4.46.3
!pip install einops addict easydict
print("✅ DeepSeek OCR bağımlılıkları kuruldu!")

✅ DeepSeek OCR bağımlılıkları kuruldu!


In [ ]:
!pip install accelerate

In [ ]:
from transformers import AutoModel, AutoTokenizer
import torch
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

model_name = "deepseek-ai/DeepSeek-OCR"

print(f"🚀 DeepSeek OCR modeli yükleniyor: {model_name}")

# Tokenizer yükle
deepseek_tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

deepseek_model = AutoModel.from_pretrained(
    model_name,
    _attn_implementation="eager",
    trust_remote_code=True,
    use_safetensors=True
)
deepseek_model = deepseek_model.eval().cuda().to(torch.bfloat16)

print("✅ DeepSeek OCR modeli yüklendi!")

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

🚀 DeepSeek OCR modeli yükleniyor: deepseek-ai/DeepSeek-OCR


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
A new version of the following files was downloaded from https://huggingface.co/deepseek-ai/DeepSeek-OCR:
- configuration_deepseek_v2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/deepseek-ai/DeepSeek-OCR:
- modeling_deepseekv2.py
- deepencoder.py
- conversation.py
. Make sure to double-check t

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-000001.safetensors:   0%|          | 0.00/6.67G [00:00<?, ?B/s]

Some weights of DeepseekOCRForCausalLM were not initialized from the model checkpoint at deepseek-ai/DeepSeek-OCR and are newly initialized: ['model.vision_model.embeddings.position_ids']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ DeepSeek OCR modeli yüklendi!


In [ ]:
def deepseek_ocr(image_path, output_dir="/content/ocr_output"):
    """
    DeepSeek OCR ile görüntüden metin çıkarır.

    Args:
        image_path: Görüntü dosya yolu
        output_dir: Çıktı klasörü

    Returns:
        Çıkarılan metin (markdown formatında)
    """
    # OCR prompt
    prompt = "<image>\n<|grounding|>Convert the document to markdown. "

    # İnference
    # Boyut seçenekleri:
    # Tiny: base_size=512, image_size=512, crop_mode=False
    # Small: base_size=640, image_size=640, crop_mode=False
    # Base: base_size=1024, image_size=1024, crop_mode=False
    # Large: base_size=1280, image_size=1280, crop_mode=False
    # Gundam: base_size=1024, image_size=640, crop_mode=True

    os.makedirs(output_dir, exist_ok=True)

    result = deepseek_model.infer(
        deepseek_tokenizer,
        prompt=prompt,
        image_file=image_path,
        output_path=output_dir,
        base_size=1024,
        image_size=640,
        crop_mode=True,
        save_results=False,
        test_compress=False
    )

    return result

print("✅ deepseek_ocr fonksiyonu hazır!")

✅ deepseek_ocr fonksiyonu hazır!


In [ ]:
# DeepSeek OCR Test
import os
test_image = "/content/drive/MyDrive/deep-learning/rvlcdip_split_v3/test/form/0001215128.tif"

print(f"Dosya: {os.path.basename(test_image)}")
print("OCR isleniyor...")

result = deepseek_ocr(test_image)


print("DEEPSEEK OCR SONUC:")

print(result)


Dosya: 0001215128.tif
OCR isleniyor...


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.
`get_max_cache()` is deprecated for all Cache classes. Use `get_max_cache_shape()` instead. Cal

BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([6, 100, 1280])
<|ref|>text<|/ref|><|det|>[[370, 238, 550, 252]]<|/det|>
TED DATES & COMPANY, INC. 

<|ref|>text<|/ref|><|det|>[[323, 264, 586, 278]]<|/det|>
BROWN & WILLIAMSON TOBACCO CORPORATION 

<|ref|>text<|/ref|><|det|>[[370, 280, 555, 293]]<|/det|>
PRINT PRODUCTION ESTIMATE 

<|ref|>text<|/ref|><|det|>[[159, 308, 714, 323]]<|/det|>
BRAND KOOL DATE NOVEMBER 24, 1970 

<|ref|>text<|/ref|><|det|>[[159, 330, 714, 345]]<|/det|>
CAPTION EST. # JQ-92-71-1 

<|ref|>text<|/ref|><|det|>[[159, 350, 714, 365]]<|/det|>
SUBJECT SEE BELOW (MEST COAST SHOOTING) B&W CODE # 

<|ref|>text<|/ref|><|det|>[[159, 370, 714, 385]]<|/det|>
PHOTOGRAPHER/ARTIST JOB # K-4099 

<|ref|>text<|/ref|><|det|>[[205, 405, 711, 420]]<|/det|>
PUBLICATION(S) COVER DATE(S) SIZE(S) COLOR 

<|ref|>text<|/ref|><|det|>[[134, 420, 700, 435]]<|/det|>
3 BLACK ADS "NICIPLE," "BOAT," WATCHFULLY 1971 9 3/8 X 12 1/8 4/0 

<|ref|>text<|/ref|><|det|>[[134, 433, 300, 444]]<|/det

Test edilen örneklerde, yukarıdaki örnekte de olduğu gibi, Deepseekocr modelinin de tutarlı bir sonuç vermediği çoğu zaman görüntünün sonunda tekrar tekrar metni yazdığı görülmüştür.